In [3]:
import torch
import pandas as pd

In [4]:
df = pd.read_csv("IMDB-Dataset.csv")

In [5]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
df.shape

(50000, 2)

In [7]:
df.groupby('sentiment')['review'].count()

sentiment
negative    25000
positive    25000
Name: review, dtype: int64

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   review     50000 non-null  str  
 1   sentiment  50000 non-null  str  
dtypes: str(2)
memory usage: 63.6 MB


In [9]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [10]:
df['review'].duplicated().sum()

np.int64(418)

In [11]:
duplicated = df[df.duplicated()]

In [12]:
duplicated.shape

(418, 2)

In [13]:
new_df = df[df.duplicated() == False]

In [14]:
new_df.shape

(49582, 2)

In [15]:
reviews, labels = df['review'], df['sentiment']

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (accuracy_score, classification_report, f1_score, precision_score, recall_score)

In [24]:
x_train, X, y_train, Y = train_test_split(reviews, labels, test_size = 0.3, random_state = 42, stratify = labels)
x_val, x_test, y_val, y_test = train_test_split(X, Y, test_size = 0.5, random_state = 42, stratify = Y)
print(x_train.shape)
print(x_val.shape)
print(x_test.shape)

(35000,)
(7500,)
(7500,)


In [25]:
common = len(set(x_train) & set(x_test)) + len(set(x_train) & set(x_val)) + len(set(x_test) & set(x_val))
print(common)

191


In [26]:
tfidf = TfidfVectorizer()
x_train_vec = tfidf.fit_transform(x_train)
x_val_vec = tfidf.transform(x_val)
x_test_vec = tfidf.transform(x_test)

In [27]:
model = LogisticRegression()
model = model.fit(x_train_vec, y_train)
y_pred = model.predict(x_val_vec)
cr = classification_report(y_val, y_pred)
print(cr)
print(f"Precision : {precision_score(y_val,y_pred, pos_label="positive")}")
print(f"Recall : {recall_score(y_val,y_pred, pos_label="positive")}")
print(f"Accuracy : {accuracy_score(y_val,y_pred)}")

              precision    recall  f1-score   support

    negative       0.90      0.88      0.89      3750
    positive       0.89      0.90      0.89      3750

    accuracy                           0.89      7500
   macro avg       0.89      0.89      0.89      7500
weighted avg       0.89      0.89      0.89      7500

Precision : 0.885638998682477
Recall : 0.8962666666666667
Accuracy : 0.8902666666666667


In [28]:
words = ["cat", "dog", "cat", "bird"]
vocabulary = {word: i for i, word in enumerate(set(words))}

print(vocabulary)

{'dog': 0, 'cat': 1, 'bird': 2}


In [29]:
label_map = {"positive" : 1, "negative" : 0}
X,Y = new_df['review'], [label_map[sentiment] for sentiment in new_df['sentiment']]

In [30]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from transformers import DataCollatorWithPadding
import torch.nn as nn

In [31]:
class ImdbDataset(Dataset):
    def __init__(self, tokenized_data, labels):
        self._input_ids = tokenized_data['input_ids']
        self._attention_mask = tokenized_data['attention_mask']
        self._labels = labels
    def __len__(self):
        return len(self._input_ids)
    def __getitem__(self, idx):
        return {
                "input_ids": torch.tensor(self._input_ids[idx]),
                "attention_mask": torch.tensor(self._attention_mask[idx]),
                "labels": torch.tensor(self._labels[idx])
                }

In [32]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize_data(X):
    tokenized_data = {"input_ids" : [], "attention_mask" : []}
    for start in range(0, len(X), 500):
        batch = X[start:start + 500]
        tokenized_batch = tokenizer(
            batch.tolist(),
            truncation=True,
            max_length=256,
            padding=True
        )
        tokenized_data['input_ids'].extend(tokenized_batch['input_ids'])
        tokenized_data['attention_mask'].extend(tokenized_batch['attention_mask'])
    return tokenized_data

In [33]:
def loader(X,Y):
    X_train, X_rest, Y_train,Y_rest = train_test_split(X,Y, test_size = 0.3, random_state = 42, stratify = Y)
    X_val, X_test,Y_val, Y_test = train_test_split(X_rest, Y_rest, test_size = 0.5, random_state = 42, stratify = Y_rest)

    X_train_tokenized = tokenize_data(X_train)
    X_val_tokenized = tokenize_data(X_val)
    X_test_tokenized = tokenize_data(X_test)

    X_train_dataset = ImdbDataset(X_train_tokenized, Y_train)
    X_val_dataset = ImdbDataset(X_val_tokenized, Y_val)
    X_test_dataset = ImdbDataset(X_test_tokenized, Y_test)

    collate_fn = DataCollatorWithPadding(tokenizer, padding=True)

    X_train_loader = DataLoader(X_train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)
    X_val_loader = DataLoader(X_val_dataset, batch_size=64, shuffle = False, collate_fn=collate_fn)
    X_test_loader = DataLoader(X_test_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

    return X_train_loader, X_val_loader, X_test_loader
    

In [34]:
X_train_loader, X_val_loader, X_test_loader = loader(X,Y)

In [35]:
batch = next(iter(X_train_loader))

for key, value in batch.items():
    print(key, value.shape)

input_ids torch.Size([64, 256])
attention_mask torch.Size([64, 256])
labels torch.Size([64])


In [36]:
model_name="distilbert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels = 2)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2307.86it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [37]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [38]:
model = model.to(device)

In [39]:
print(next(model.parameters()).device)

cuda:0


In [40]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

In [41]:
from tqdm.auto import tqdm

In [42]:
def training():
    model.train()
    total_loss = 0
    batch_count = 0
    progress_bar = tqdm(X_train_loader, desc = "Training")
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = output.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        batch_count += 1
        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            avg_loss=f"{total_loss/batch_count:.4f}"
        )
    print("Average Training loss:",total_loss/len(X_train_loader))

In [43]:
def validation():
    model.eval()
    batch_count=0
    total_val_loss=0
    correct=0
    with torch.no_grad():
        progress_bar = tqdm(X_val_loader, desc="Training")
        for batch in progress_bar:
            input_ids=batch['input_ids'].to(device)
            attention_mask=batch['attention_mask'].to(device)
            labels=batch['labels'].to(device)
            output = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss=output.loss
            total_val_loss+=loss.item()
            batch_count+=1
            correct += (output.logits.argmax(dim=1) == labels).sum().item()
            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}",
                avg_loss=f"{total_val_loss/batch_count:.4f}"
            )
    print("Average validation loss:",total_val_loss/len(X_val_loader))
    print("Validation accuracy:",correct/len(X_val_loader.dataset))

In [44]:
best_val_accuracy = 0
for epoch in range(0):
    print(f"Epoch: {epoch + 1}")
    model.train()
    total_loss = 0
    batch_count = 0
    correct=0
    progress_bar = tqdm(X_train_loader, desc = "Training")
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = output.loss
        correct += (output.logits.argmax(dim=1)==labels).sum().item()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        batch_count += 1
        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            avg_loss=f"{total_loss/batch_count:.4f}"
        )
    print("Average Training loss:",total_loss/len(X_train_loader))
    print("Training accuracy:",correct/len(X_train_loader.dataset))

    model.eval()
    batch_count=0
    total_val_loss=0
    correct=0
    with torch.no_grad():
        progress_bar = tqdm(X_val_loader, desc="Validation")
        for batch in progress_bar:
            input_ids=batch['input_ids'].to(device)
            attention_mask=batch['attention_mask'].to(device)
            labels=batch['labels'].to(device)
            output = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss=output.loss
            total_val_loss+=loss.item()
            batch_count+=1
            correct += (output.logits.argmax(dim=1) == labels).sum().item()
            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}",
                avg_loss=f"{total_val_loss/batch_count:.4f}"
            )
    val_accuracy = correct/len(X_val_loader.dataset)
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        model.save_pretrained("best_imdb_model")
        tokenizer.save_pretrained("best_imdb_model")
    print("Average validation loss:",total_val_loss/len(X_val_loader))
    print("Validation accuracy:",val_accuracy)

In [47]:
loaded_model = AutoModelForSequenceClassification.from_pretrained("best_imdb_model")
loaded_tokenizer = AutoTokenizer.from_pretrained("best_imdb_model")
loaded_model.to(device)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 4522.21it/s]


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [57]:
predictions = []
actual_labels = []
def evaluation():
    loaded_model.eval()
    test_loss = 0
    batch_size = 0
    correct = 0
    with torch.no_grad():
        progress_bar = tqdm(X_test_loader, desc = "Testing")
        for batch in progress_bar:
            batch_size += 1
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            output = loaded_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = output.loss
            test_loss += loss.item()
            pred = (output.logits.argmax(dim=1))
            predictions.extend(pred.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())
            correct += (pred == labels).sum().item()
            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}",
                avg_loss=f"{test_loss/batch_size:.4f}"
            )
    accuracy = correct/len(X_test_loader.dataset)
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test Loss: {test_loss/batch_size:.4f}")

evaluation()
            

Testing: 100%|██████████| 117/117 [00:37<00:00,  3.09it/s, avg_loss=0.2282, loss=0.1380]

Test Accuracy: 0.9066
Test Loss: 0.2282


In [65]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

print(classification_report(
    actual_labels,
    predictions,
    target_names=["Negative", "Positive"]
))

print(confusion_matrix(actual_labels, predictions))
print(f"F1 score: {f1_score(actual_labels, predictions, pos_label=1)}")
print(f"F1 score: {precision_score(actual_labels, predictions, pos_label=1)}")
print(f"F1 score: {recall_score(actual_labels, predictions, pos_label=1)}")

              precision    recall  f1-score   support

    Negative       0.90      0.91      0.91      3705
    Positive       0.91      0.90      0.91      3733

    accuracy                           0.91      7438
   macro avg       0.91      0.91      0.91      7438
weighted avg       0.91      0.91      0.91      7438

[[3385  320]
 [ 375 3358]]
F1 score: 0.9062204830657131
F1 score: 0.9129961935834693
F1 score: 0.8995446021966247
